# L4 Taxonomy Classification (Anchor-Based)

Classifies product and service listings into the 13 unified L4 taxonomy categories using
cosine similarity between listing embeddings and taxonomy anchor embeddings.

**Approach:** Embed L4 anchor descriptions → compute cosine similarity against cached listing
embeddings → assign highest-scoring L4 → flag low-confidence items for residual clustering.

**Modes:**
- `validate_products`: Load from LEI products table. Validates anchor descriptions by checking
  whether known products land on expected L4s. Use this to calibrate `CONFIDENCE_THRESHOLD`
  before classifying services.
- `classify_services`: Load services product buckets from the SERVICES table. Runs the full
  classification pipeline.

**Prerequisites:**
- `pipeline/taxonomy/l3_taxonomy_anchors.json` — L4 anchor definitions (finalize descriptions before running)
- Titan embedding cache populated for the target dataset (LEI run must complete before services run)


In [28]:
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    raise FileNotFoundError("Could not locate project root containing product_classifier_utils.py")

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from product_classifier_utils import (
    build_product_text,
    embed_texts_from_cache,
    get_bedrock_client,
    get_snowflake_session,
    stable_text_hash,
)

In [29]:
# ── Run mode ─────────────────────────────────────────────────────────────────
# "validate_products" : Load LEI products table. Use for threshold calibration.
# "classify_services" : Load services product buckets for full classification.
# "classify_lcg"      : Load LCG products table (~400K). Same pipeline as services.
MODE = "classify_services"

# ── Snowflake tables ──────────────────────────────────────────────────────────
PRODUCTS_TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PRODUCTS_LEI"
SERVICES_TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.SERVICES"
LCG_TABLE      = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PRODUCTS_LCG"

# Which service buckets to include in classify_services mode.
# These are the product-heavy buckets identified from the data analysis.
# Set to None to load all services (not recommended — includes genuine services).
SERVICE_BUCKETS = [
    # (PARENT_3_CATEGORY, PARENT_4_CATEGORY or None for blank)
    ("Chemistry and Materials", None),          # 4.84M blank-L4 chemicals
    ("Compounds", None),                        # 747K raw compounds
    ("Biology", "Cells and Tissues"),           # 570K misclassified products
    ("Biology", "Biochemistry & Molecular Biology"),  # 2.93M ambiguous
]

# Optionally limit rows for development / testing
ROW_LIMIT = None  # set to e.g. 10_000 for a quick test run

# ── Embedding + AWS ───────────────────────────────────────────────────────────
AWS_PROFILE = "staging.admin"
AWS_REGION = "us-east-1"
MODEL_ID = "amazon.titan-embed-text-v1"
MAX_WORKERS = 16
CHECKPOINT_EVERY = 50000  # Now safe — checkpoints only write new embeddings, not full cache
MAX_RETRIES = 5
CACHE_PATH = PROJECT_ROOT / "artifacts/cache/embedding_cache.pkl"

# ── Classification ────────────────────────────────────────────────────────────
ANCHORS_PATH = PROJECT_ROOT / "pipeline/taxonomy/l3_taxonomy_anchors.json"

# Confidence threshold (absolute cosine similarity) — kept for histogram reference only.
CONFIDENCE_THRESHOLD = 0.50

# Margin threshold: gap between top-1 and top-2 cosine scores.
# Listings below this are genuinely ambiguous between two L4s and go to residual.
# Titan absolute scores vary by data type (chemicals score lower than equipment),
# but margin is data-agnostic and consistent across LEI, Services, and LCG.
MARGIN_THRESHOLD = 0.05  # ~68% confident, ~32% residual on services data

# Whether to embed texts missing from cache. Set False to do a dry-run that
# reports cache coverage without calling Titan (useful for scoping overnight jobs).
RUN_EMBEDDING = True

# ── Output ────────────────────────────────────────────────────────────────────
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "analysis"
RUN_TAG = f"l4_classification_{MODE}"
SAVE_TO_SNOWFLAKE = False
OUTPUT_TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.L4_CLASSIFICATIONS"

## 1. Load L4 Taxonomy Anchors

In [30]:
with open(ANCHORS_PATH) as f:
    anchors_data = json.load(f)

anchors = anchors_data["anchors"]
anchor_ids    = [a["id"]          for a in anchors]
anchor_labels = [a["label"]       for a in anchors]
anchor_texts  = [a["description"] for a in anchors]

print(f"Loaded {len(anchors)} L4 taxonomy anchors:")
for i, (aid, label) in enumerate(zip(anchor_ids, anchor_labels)):
    print(f"  {i:2d}. [{aid}] {label}")

Loaded 13 L4 taxonomy anchors:
   0. [chemicals_solvents] Chemicals & Solvents
   1. [molecular_biology_reagents] Molecular Biology Reagents
   2. [lab_supplies_consumables] Lab Supplies & Consumables
   3. [antibodies] Antibodies
   4. [proteins_peptides] Proteins & Peptides
   5. [kits_assays] Kits & Assays
   6. [cell_tissue_culture] Cell & Tissue Culture
   7. [biospecimens] Biospecimens
   8. [animal_models] Animal Models
   9. [equipment_instruments_parts] Equipment, Instruments & Parts
  10. [controls_calibrators_standards] Controls, Calibrators & Standards
  11. [furniture_storage] Furniture & Storage
  12. [general_office_supplies] General Office Supplies


## 2. Connect to Snowflake and Load Listings

In [31]:
sf_session = get_snowflake_session()

def _build_services_where(buckets):
    """Build WHERE clause for the service product bucket filters."""
    clauses = []
    for l3, l4 in buckets:
        if l4 is None:
            clauses.append(
                f"(PARENT_3_CATEGORY = '{l3}' AND "
                f"(PARENT_4_CATEGORY IS NULL OR TRIM(PARENT_4_CATEGORY) = ''))"
            )
        else:
            clauses.append(
                f"(PARENT_3_CATEGORY = '{l3}' AND PARENT_4_CATEGORY = '{l4}')"
            )
    return "(\n  " + "\n  OR ".join(clauses) + "\n)"


if MODE == "validate_products":
    query = f"""
    SELECT
        PRODUCT_ID,
        PRODUCT_NAME,
        DESCRIPTION,
        PRICING_STATUS_C,
        LIST_PRICE_C,
        PARENT_3_CATEGORY AS CURRENT_L3
    FROM {PRODUCTS_TABLE}
    WHERE UPPER(COALESCE(DESCRIPTION, '')) NOT LIKE '%INSERT%'
    """
    if ROW_LIMIT:
        query += f"\nLIMIT {int(ROW_LIMIT)}"

elif MODE == "classify_services":
    # SERVICES table is pre-filtered upstream in Snowflake to the 7M priced listings of interest.
    query = f"""
    SELECT
        PRODUCT_ID,
        PRODUCT_NAME,
        DESCRIPTION,
        PRICING_STATUS_C,
        LIST_PRICE_C,
        PARENT_3_CATEGORY AS CURRENT_L3,
        PARENT_4_CATEGORY AS CURRENT_L4,
        PARENT_5_CATEGORY AS CURRENT_L5
    FROM {SERVICES_TABLE}
    """
    if ROW_LIMIT:
        query += f"\nLIMIT {int(ROW_LIMIT)}"

elif MODE == "classify_lcg":
    query = f"""
    SELECT
        PRODUCT_ID,
        PRODUCT_NAME,
        DESCRIPTION,
        PRICING_STATUS_C,
        LIST_PRICE_C,
        PARENT_3_CATEGORY AS CURRENT_L3,
        PARENT_4_CATEGORY AS CURRENT_L4,
        PARENT_5_CATEGORY AS CURRENT_L5
    FROM {LCG_TABLE}
    """
    if ROW_LIMIT:
        query += f"\nLIMIT {int(ROW_LIMIT)}"

else:
    raise ValueError(f"Unknown MODE: {MODE!r}. Must be 'validate_products', 'classify_services', or 'classify_lcg'.")

print(f"Query (MODE={MODE}):")
print(query)

df = sf_session.sql(query).to_pandas()
print(f"\nLoaded rows: {len(df):,}")
print(df.head(3))

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://scienceexchange.okta.com/app/snowflake/exktzixqmxM7e7kdB697/sso/saml?SAMLRequest=lZLdb5swFMX%2FFeQ9gyHfsZJUaVG0SO2aJqST%2BuaZm8QDbOprAu1fP5OPqXtopb0hc45%2Fx%2FfcyU1T5N4RDEqtpiQKQuKBEjqVaj8l22Thj4iHlquU51rBlLwBkpvZBHmRl2xe2YNaw2sFaD13kULW%2FpiSyiimOUpkiheAzAq2mT%2Fcs04QMo4IxjocuVhSlI51sLZklNZ1HdTdQJs97YRhSMMxdapW8o18QJRfM0qjrRY6v1oa96ZPEBENey3CKRxhdTHeSnUewVeUX2cRsu9JsvJXj5uEePPr6%2B60wqoAswFzlAK26%2FtzAHQJykz0h73%2BKKjQB47WjwJUut7lPAOhi7Ky7trAfdEdpDTXe%2BmGtYynpMxkmvDsHZ6PqHkxLszvxVNvcOyv458bG69L1d2%2B9LkQ9dM%2BFiNBvOdrtZ222iViBUvVFmrdUdgZ%2BGHfj4ZJNGa9Eet2gqg7eCFe7AqVituT85oahXRDAmjEgas9BDqz%2FBSSlyX9m59Ck9l32bwWzcMQhll6OxgPKaKmbW%2FkvDrsFMTM%2FnsgE%2FrRflnDH66ZZbzSuRRv3kKbgtvPi4uC6HQiU393kjIouMznaWoA0RWY57q%2BM8Ct23ZrKiB0dqb%2Bu%2B%2BzPw%3D%3D&RelayState=ver%3A3-hint%3A9376132683223046-ETMsDgAAAZ43e6sIABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEFzwQcq2rZGbjOJGKrt9iXUAAACg3R2Ep

KeyboardInterrupt: 

## 3. Load Embedding Cache

In [ ]:
def save_cache(cache: dict, path: Path) -> None:
    """Atomic cache write: write to temp file then rename to avoid corruption on interrupt."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix('.pkl.tmp')
    with open(tmp_path, "wb") as f:
        pickle.dump(cache, f, protocol=pickle.HIGHEST_PROTOCOL)
    tmp_path.replace(path)  # atomic on same filesystem
    print(f"Cache saved: {len(cache):,} entries → {path}")


# Load only the key index (350MB) instead of the full 32GB cache.
# Run extract_cache_keys.py once in a terminal to generate this file.
KEYS_PATH = CACHE_PATH.with_name("embedding_cache_keys.pkl")
if KEYS_PATH.exists():
    with open(KEYS_PATH, "rb") as f:
        embedding_cache_keys = pickle.load(f)
    print(f"Key index loaded: {len(embedding_cache_keys):,} hashes ({KEYS_PATH.stat().st_size/1e6:.0f} MB)")
    embedding_cache = None  # full cache not loaded — using key index only
elif CACHE_PATH.exists():
    print("WARNING: Key index not found. Run extract_cache_keys.py first to avoid OOM crashes.")
    print("Falling back to loading full cache (may crash on machines with < 40GB RAM).")
    with open(CACHE_PATH, "rb") as f:
        embedding_cache = pickle.load(f)
    embedding_cache_keys = set(embedding_cache.keys())
    print(f"Cache loaded: {len(embedding_cache_keys):,} entries from {CACHE_PATH}")
else:
    embedding_cache = {}
    embedding_cache_keys = set()
    print(f"No cache found at {CACHE_PATH} — starting fresh.")

## 4. Embed L4 Anchor Descriptions

Only 13 embeddings — completes in seconds.

In [ ]:
bedrock_client = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)

anchor_hashes = [stable_text_hash(t) for t in anchor_texts]

# Use a small dedicated cache for the 13 anchors — does not need the full volume 1/2 caches
ANCHOR_CACHE_PATH = CACHE_PATH.with_name("anchor_cache.pkl")
if ANCHOR_CACHE_PATH.exists():
    with open(ANCHOR_CACHE_PATH, "rb") as f:
        anchor_cache = pickle.load(f)
    print(f"Anchor cache loaded: {len(anchor_cache)} entries")
else:
    anchor_cache = {}

anchor_matrix = embed_texts_from_cache(
    texts=anchor_texts,
    text_hashes=anchor_hashes,
    cache=anchor_cache,
    client=bedrock_client,
    model_id=MODEL_ID,
    show_progress=True,
    max_workers=1,
    max_retries=MAX_RETRIES,
)

print(f"Anchor matrix shape: {anchor_matrix.shape}")

# L2-normalize anchors for cosine similarity via dot product
anchor_norms = np.linalg.norm(anchor_matrix, axis=1, keepdims=True)
anchor_matrix_normed = anchor_matrix / np.clip(anchor_norms, 1e-10, None)
print("Anchor embeddings normalised.")

# Persist the 13 anchor embeddings to their own small cache file
save_cache(anchor_cache, ANCHOR_CACHE_PATH)

## 5. Build Listing Texts and Compute Embeddings

For the validate_products run these will already be cached (LEI embedding run).
For classify_services, most will be missing and require the overnight embedding job.
Set `RUN_EMBEDDING = False` in config to do a dry-run that reports cache coverage only.

In [ ]:
texts = build_product_text(df).tolist()
hashes = [stable_text_hash(t) for t in texts]

missing_hashes = [h for h in set(hashes) if h not in embedding_cache_keys]
cache_hits = len(set(hashes)) - len(missing_hashes)
cache_pct = cache_hits / len(set(hashes)) * 100 if hashes else 0

print(f"Total listings:         {len(df):,}")
print(f"Unique text hashes:     {len(set(hashes)):,}")
print(f"Cache hits:             {cache_hits:,} ({cache_pct:.1f}%)")
print(f"Missing (need embed):   {len(missing_hashes):,}")

if missing_hashes and not RUN_EMBEDDING:
    print("\nRUN_EMBEDDING=False — stopping here. Set RUN_EMBEDDING=True to embed missing texts.")
    print("Run this as an overnight job for large missing counts.")

In [ ]:
# ── Incremental cache: new embeddings go to a SEPARATE small file ─────────
# The main embedding_cache (~32GB) is never rewritten mid-run.
# Only newly computed embeddings are checkpointed — fast, small writes.
NEW_CACHE_PATH = CACHE_PATH.with_name("embedding_cache_new.pkl")

if NEW_CACHE_PATH.exists():
    with open(NEW_CACHE_PATH, "rb") as f:
        new_embeddings = pickle.load(f)
    print(f"Resuming: {len(new_embeddings):,} new embeddings already in incremental cache")
else:
    new_embeddings = {}
    print("Starting fresh incremental cache")

class _MergedCache(dict):
    """Dict that checks base key index before reporting a miss. Writes go here only."""
    def __init__(self, base_keys, new_entries):
        super().__init__(new_entries)
        self._base_keys = base_keys  # set of hashes — no values loaded
    def __contains__(self, key):
        return super().__contains__(key) or key in self._base_keys
    def __getitem__(self, key):
        if super().__contains__(key):
            return super().__getitem__(key)
        # Value not in new_embeddings — will be looked up from full cache at classification time
        raise KeyError(f"Hash {key[:8]}... is in base cache keys but values not loaded (by design)")

merged_cache = _MergedCache(embedding_cache_keys, new_embeddings)

# Recompute missing using merged view
missing_hashes_merged = [h for h in set(hashes) if h not in merged_cache]
print(f"Missing after merging incremental cache: {len(missing_hashes_merged):,}")

if missing_hashes_merged and not RUN_EMBEDDING:
    print("RUN_EMBEDDING=False — skipping embed step.")
elif missing_hashes_merged:
    def on_checkpoint(cache, processed):
        new_only = dict(cache)
        tmp = NEW_CACHE_PATH.with_suffix('.pkl.tmp')
        with open(tmp, 'wb') as f:
            pickle.dump(new_only, f, protocol=pickle.HIGHEST_PROTOCOL)
        tmp.replace(NEW_CACHE_PATH)
        print(f"Checkpointed {len(new_only):,} new embeddings → {NEW_CACHE_PATH.name}")

    # Pass only missing texts — avoids building a huge return matrix for cached items
    text_by_hash = {stable_text_hash(t): t for t in texts}
    missing_texts_only = [text_by_hash[h] for h in missing_hashes_merged]

    embed_texts_from_cache(
        texts=missing_texts_only,
        text_hashes=missing_hashes_merged,
        cache=merged_cache,
        client=bedrock_client,
        model_id=MODEL_ID,
        show_progress=True,
        max_workers=MAX_WORKERS,
        checkpoint_every=50000,
        on_checkpoint=on_checkpoint,
        max_retries=MAX_RETRIES,
    )
    del missing_texts_only  # free memory immediately

    new_final = dict(merged_cache)
    save_cache(new_final, NEW_CACHE_PATH)
    print(f"Embedding complete. Volume 2: {len(new_final):,} new embeddings saved to {NEW_CACHE_PATH.name}")
    print("Run the classification script (classify_services_batched.py) to produce L4 assignments.")
else:
    print("All embeddings already in cache — no new Bedrock calls needed.")
    print("Run the classification script (classify_services_batched.py) to produce L4 assignments.")

listing_matrix_normed = None  # Classification is done via batched script, not in-memory matrix

## 6. Cosine Similarity Classification

Similarity matrix: `(N listings × 1536) @ (1536 × 13 anchors)` — fast matrix multiply.
Each listing is assigned to the anchor with the highest cosine similarity.

In [ ]:
# Classify listings in two phases to avoid OOM:
#   Phase A — Volume 2 listings (new_embeddings already in memory from Step 5)
#   Phase B — Volume 1 listings (classify_v1.py loads main cache separately)

output_cols = ["PRODUCT_ID", "ASSIGNED_L4_ID", "ASSIGNED_L4_LABEL",
               "CONFIDENCE", "CONFIDENCE_MARGIN", "IS_LOW_CONFIDENCE"]
if "CURRENT_L3" in df.columns: output_cols.insert(2, "CURRENT_L3")
if "CURRENT_L4" in df.columns: output_cols.insert(3, "CURRENT_L4")

output_dir = LOCAL_OUTPUT_DIR / RUN_TAG
output_dir.mkdir(parents=True, exist_ok=True)

# ── Phase A: classify v2 listings ─────────────────────────────────────────
v2_hashes_set = set(new_embeddings.keys())
v2_mask = np.array([h in v2_hashes_set for h in hashes])
v2_indices = np.where(v2_mask)[0]
print(f"Volume 2 listings (classify now): {v2_mask.sum():,}")
print(f"Volume 1 listings (classify via script): {(~v2_mask).sum():,}")

BATCH = 250_000
v2_records = []
for start in range(0, len(v2_indices), BATCH):
    idx_batch = v2_indices[start:start + BATCH]
    vecs = np.array([new_embeddings[hashes[i]] for i in idx_batch], dtype=np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    vecs_n = vecs / np.clip(norms, 1e-10, None)
    del vecs
    sim = vecs_n @ anchor_matrix_normed.T
    del vecs_n
    top1 = sim.argmax(axis=1)
    top1_s = sim[np.arange(len(sim)), top1]
    sim2 = sim.copy(); sim2[np.arange(len(sim)), top1] = -1
    margin = top1_s - sim2.max(axis=1)
    del sim, sim2
    batch_df = df.iloc[idx_batch].copy()
    batch_df["ASSIGNED_L4_ID"]    = [anchor_ids[i]    for i in top1]
    batch_df["ASSIGNED_L4_LABEL"] = [anchor_labels[i] for i in top1]
    batch_df["CONFIDENCE"]        = top1_s.round(4)
    batch_df["CONFIDENCE_MARGIN"] = margin.round(4)
    batch_df["IS_LOW_CONFIDENCE"] = top1_s < CONFIDENCE_THRESHOLD
    v2_records.append(batch_df[output_cols])
    print(f"  Batch {start:,}–{start+len(idx_batch):,} done")

v2_results = pd.concat(v2_records, ignore_index=True)
del v2_records
v2_path = output_dir / "classification_results_v2.csv"
v2_results.to_csv(v2_path, index=False)
print(f"\nPhase A saved: {v2_path} ({len(v2_results):,} rows)")

# ── Save v1 work file for the classify_v1.py script ───────────────────────
v1_mask = ~v2_mask
v1_indices = np.where(v1_mask)[0]
v1_work_cols = ["PRODUCT_ID"] + [c for c in ["CURRENT_L3","CURRENT_L4","CURRENT_L5"] if c in df.columns]
v1_work = df.iloc[v1_indices][v1_work_cols].copy()
v1_work["_HASH"] = [hashes[i] for i in v1_indices]
v1_work_path = output_dir / "v1_work.parquet"
v1_work.to_parquet(v1_work_path, index=False)
print(f"Phase B work file saved: {v1_work_path} ({len(v1_work):,} rows)")
print("\nRun classify_v1.py in terminal to complete Phase B, then run the combine cell below.")

## 7. Combine Phase A + Phase B Results

Run this cell after `classify_v1.py` has completed in the terminal.

In [ ]:
output_dir = LOCAL_OUTPUT_DIR / RUN_TAG
output_dir.mkdir(parents=True, exist_ok=True)

v2_path = output_dir / "classification_results_v2.csv"
v1_path = output_dir / "classification_results_v1.csv"

parts = []
if v2_path.exists():
    parts.append(pd.read_csv(v2_path))
    print(f"Phase A (v2): {len(parts[-1]):,} rows")
if v1_path.exists():
    parts.append(pd.read_csv(v1_path))
    print(f"Phase B (v1): {len(parts[-1]):,} rows")
else:
    print("Phase B not yet complete. Run classify_v1.py first, then re-run this cell.")

df_results = pd.concat(parts, ignore_index=True)
print(f"Combined total: {len(df_results):,} rows")

# Recompute IS_LOW_CONFIDENCE using margin threshold (consistent across all datasets)
df_results["IS_LOW_CONFIDENCE"] = df_results["CONFIDENCE_MARGIN"] < MARGIN_THRESHOLD
high = (~df_results["IS_LOW_CONFIDENCE"]).sum()
print(f"Margin threshold: {MARGIN_THRESHOLD}")
print(f"High-confidence (margin >= {MARGIN_THRESHOLD}): {high:,} ({high/len(df_results)*100:.1f}%)")
print(f"Residual (margin < {MARGIN_THRESHOLD}):          {df_results['IS_LOW_CONFIDENCE'].sum():,} ({df_results['IS_LOW_CONFIDENCE'].mean()*100:.1f}%)")

# Save combined results
combined_path = output_dir / "classification_results.csv"
df_results.to_csv(combined_path, index=False)
print(f"Combined results saved: {combined_path}")

# Distribution summary
dist = (
    df_results.groupby(["ASSIGNED_L4_LABEL", "IS_LOW_CONFIDENCE"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "HIGH_CONF", True: "LOW_CONF"})
)
dist["TOTAL"] = dist.sum(axis=1)
dist["PCT"]   = (dist["TOTAL"] / len(df_results) * 100).round(1)
dist = dist.sort_values("TOTAL", ascending=False)
print("\nL4 Assignment Distribution:")
print(dist.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
labels = dist.index.tolist()
x = range(len(labels))
high = dist["HIGH_CONF"] if "HIGH_CONF" in dist.columns else pd.Series(0, index=dist.index)
low  = dist["LOW_CONF"]  if "LOW_CONF"  in dist.columns else pd.Series(0, index=dist.index)
ax.bar(x, high, label="High confidence", color="steelblue")
ax.bar(x, low, bottom=high, label="Low confidence", color="tomato")
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Listings")
ax.set_title(f"L4 Assignment Distribution — {MODE} (threshold={CONFIDENCE_THRESHOLD})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confidence score distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_results["CONFIDENCE"], bins=100, color="steelblue", edgecolor="none")
ax.axvline(CONFIDENCE_THRESHOLD, color="tomato", linestyle="--", label=f"Threshold ({CONFIDENCE_THRESHOLD})")
ax.set_xlabel("Cosine Similarity (Top-1)")
ax.set_ylabel("Listings")
ax.set_title("Confidence Score Distribution")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nConfidence percentiles:")
for pct in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{pct:2d}: {np.percentile(df_results['CONFIDENCE'], pct):.4f}")

## 8. Validation Check (validate_products mode)

Spot-check assignment quality. For known product types, verify they land on
expected L4s. If common product types are landing in wrong buckets, adjust
anchor descriptions in `l3_taxonomy_anchors.json` and re-run.

In [ ]:
if MODE == "validate_products" and "CURRENT_L3" in df_results.columns:
    print("Current L3 → Assigned L4 cross-tab (counts):")
    cross = pd.crosstab(
        df_results["CURRENT_L3"],
        df_results["ASSIGNED_L4_LABEL"],
        margins=True,
    )
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print(cross)
else:
    print("Validation cross-tab only runs in validate_products mode.")

In [ ]:
# Sample 10 high-confidence and 10 low-confidence listings for manual review
print("=== 10 HIGH-CONFIDENCE SAMPLES ===")
high_sample = df_results[~df_results["IS_LOW_CONFIDENCE"]].sample(min(10, (~df_results["IS_LOW_CONFIDENCE"]).sum()), random_state=42)
for _, row in high_sample.iterrows():
    print(f"  [{row['CONFIDENCE']:.3f}] {row['ASSIGNED_L4_LABEL']:<35} | {row.get('PRODUCT_ID','')}")

print("\n=== 10 LOW-CONFIDENCE SAMPLES ===")
if df_results["IS_LOW_CONFIDENCE"].any():
    low_sample = df_results[df_results["IS_LOW_CONFIDENCE"]].sample(min(10, df_results["IS_LOW_CONFIDENCE"].sum()), random_state=42)
    for _, row in low_sample.iterrows():
        print(f"  [{row['CONFIDENCE']:.3f}] {row['ASSIGNED_L4_LABEL']:<35} | {row.get('PRODUCT_ID','')}")
else:
    print("  No low-confidence items at this threshold.")

## 9. Residual Analysis

Low-confidence listings are candidates for residual clustering to discover
missing L4 categories. Inspect before sending to the clustering notebook.

In [ ]:
residual = df_results[df_results["IS_LOW_CONFIDENCE"]].copy()
print(f"Residual size: {len(residual):,} listings ({len(residual)/len(df_results)*100:.1f}% of total)")

if len(residual) > 0:
    print("\nTop L4s assigned to residual (where assignment was uncertain):")
    print(residual["ASSIGNED_L4_LABEL"].value_counts().head(10).to_string())

    if "CURRENT_L3" in residual.columns:
        print("\nCurrent L3 breakdown of residual:")
        print(residual["CURRENT_L3"].value_counts().head(10).to_string())

## 10. Save Results

In [ ]:
# Ensure these are defined even if step 6 was skipped
output_cols = ["PRODUCT_ID", "ASSIGNED_L4_ID", "ASSIGNED_L4_LABEL",
               "CONFIDENCE", "CONFIDENCE_MARGIN", "IS_LOW_CONFIDENCE"]
for _col in ["CURRENT_L3", "CURRENT_L4"]:
    if _col in df_results.columns and _col not in output_cols:
        output_cols.insert(2, _col)
residual = df_results[df_results["IS_LOW_CONFIDENCE"]].copy()

# Save residual separately for clustering notebook
residual_cols = [c for c in output_cols if c in df_results.columns]
if len(residual) > 0:
    residual_path = output_dir / "residual_for_clustering.csv"
    residual[residual_cols].to_csv(residual_path, index=False)
    print(f"Residual saved: {residual_path} ({len(residual):,} rows)")

# Save run summary
summary = {
    "mode": MODE,
    "total_listings": len(df_results),
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "high_confidence_count": int((~df_results["IS_LOW_CONFIDENCE"]).sum()),
    "low_confidence_count": int(df_results["IS_LOW_CONFIDENCE"].sum()),
    "l4_distribution": df_results["ASSIGNED_L4_LABEL"].value_counts().to_dict(),
    "confidence_percentiles": {
        str(p): float(np.percentile(df_results["CONFIDENCE"], p))
        for p in [10, 25, 50, 75, 90, 95, 99]
    },
}
with open(output_dir / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved: {output_dir / 'run_summary.json'}")

In [ ]:
if SAVE_TO_SNOWFLAKE:
    from snowflake.snowpark import functions as F

    out_df = df_results[["PRODUCT_ID", "ASSIGNED_L4_ID", "ASSIGNED_L4_LABEL", "CONFIDENCE", "IS_LOW_CONFIDENCE"]].copy()
    out_df.columns = [c.upper() for c in out_df.columns]

    sp_df = sf_session.create_dataframe(out_df)
    sp_df.write.mode("overwrite").save_as_table(OUTPUT_TABLE)
    print(f"Written {len(out_df):,} rows to {OUTPUT_TABLE}")
else:
    print("Snowflake write skipped (SAVE_TO_SNOWFLAKE=False).")